In [ ]:
from glob import glob
import pandas as pd
import os
import pyreadr
from tqdm import tqdm

In [ ]:
file_path = "/nas/houce/UK_mobility/output_data/merged_202103_202203.csv"
data_counts = pd.read_csv(file_path)

In [ ]:
time1 = file_path.split('/')[-1].split('_')[-2]
time2 = file_path.split('/')[-1].split('_')[-1].split('.')[0]
print(time1, time2)

In [ ]:
cols_1 = [col for col in data_counts.columns if col.startswith(f'hours_present_{time1}')]
cols_2 = [col for col in data_counts.columns if col.startswith(f'hours_present_{time2}')]
filtered = data_counts[(data_counts[cols_1].gt(11).any(axis=1)) & (data_counts[cols_2].gt(11).any(axis=1))]
time_all = [i.split('_')[-1] for i in filtered.columns[1:]]
for j, time_ in tqdm(enumerate(time_all[:1]), total=len(time_all)):
    print(f"Processing time: {time_}")
    data = pd.read_csv(f"/nas/houce/UK_mobility/processed_data/filtered_data_{time_}.csv")

    id_list = filtered[filtered[f'hours_present_{time_}'].notna()]['device_aid'].tolist()
    data_filtered = data[data['device_aid'].isin(id_list)]
    data_filtered_group = data_filtered.groupby('device_aid')

    for i, (device_id, df) in tqdm(enumerate(data_filtered_group), total=len(data_filtered_group)):


        df['duration'] = 60 / df.groupby('hours')['hours'].transform('count')
        df['duration'] = df['duration'].round(2)
        df_merged = merge_duplicates(df)
        print(f"User {i}, ID: {device_id}, Original records: {len(df)}, After merging: {len(df_merged)}")
        df_merged['status'] = 'moving'  # Default all to moving

        # Get previous row's coordinates (shift)
        df_merged['prev_latitude'] = df_merged['latitude'].shift()
        df_merged['prev_longitude'] = df_merged['longitude'].shift()
        df_merged.loc[(df_merged['duration'] > 5), 'status'] = 'poi'
        # If coordinates are the same as previous hour, mark as 'poi'
        df_merged.loc[(df_merged['latitude'] == df_merged['prev_latitude']) & 
            (df_merged['longitude'] == df_merged['prev_longitude']), 'status'] = 'poi'

        # Drop the temporary columns used for calculation
        df_merged = df_merged.drop(columns=['prev_latitude', 'prev_longitude'])
        # df_merged = df_merged.drop(df_merged.index[0])
        df_merged = df_merged[df_merged['status'] != 'moving']
        location_poi_label = pd.DataFrame(df_merged[df_merged['status'] == 'poi'].value_counts(['latitude','longitude'])).reset_index()
        location_poi_label['label'] = 'amenity'
        location_poi_label.loc[0, 'label'] = 'home'
        location_poi_label.loc[1, 'label'] = 'workplace'
        df_merged = df_merged.merge(location_poi_label, on=['latitude', 'longitude'], how='left')
        # df_merged = df_merged.drop_duplicates(subset=['latitude', 'longitude', 'label'])
        if i == 0: 
            df_merged_all = df_merged
        else:
            df_merged_all = pd.concat([df_merged_all, df_merged], ignore_index=True)
        if i == 1:
            break
       
        # df_merged = df_merged.

In [ ]:
df_merged['majority_label'] = (
    df_merged.groupby('hours')['label']
    .transform(lambda x: x.value_counts().idxmax())
)

In [ ]:
tem = df_merged_all[df_merged_all['device_aid'] == 'REDACTED_DEVICE_ID']
tem['majority_label'] = (
    tem.groupby('hours')['label']
    .transform(lambda x: x.value_counts().idxmax())
)

In [ ]:
def merge_duplicates(df):
    # 按小时分组
    merged = []
    for hour, group in df.groupby('hours'):
        # 保留原始顺序
        group = group.reset_index(drop=True)
        i = 0
        while i < len(group):
            current_row = group.iloc[i].copy()
            current_lat, current_lon = current_row['latitude'], current_row['longitude']
            
            # 计算连续重复的记录数
            count = 1
            j = i + 1
            while j < len(group) and group.iloc[j]['latitude'] == current_lat and group.iloc[j]['longitude'] == current_lon:
                count += 1
                j += 1
            
            # 调整duration（乘以重复条数）
            current_row['duration'] = current_row['duration'] * count
            merged.append(current_row)
            
            # 跳过已处理的连续记录
            i = j
            
    return pd.DataFrame(merged)

df = aaaa[0][1]
df['duration'] = 60 / df.groupby('hours')['hours'].transform('count')
df['duration'] = df['duration'].round(2)
print(df.shape)
df_merged = merge_duplicates(df)
print(df_merged.shape)
df_merged['status'] = 'moving'  # Default all to moving

# Get previous row's coordinates (shift)
df_merged['prev_latitude'] = df_merged['latitude'].shift()
df_merged['prev_longitude'] = df_merged['longitude'].shift()
df_merged.loc[(df_merged['duration'] > 5), 'status'] = 'poi'
# If coordinates are the same as previous hour, mark as 'poi'
df_merged.loc[(df_merged['latitude'] == df_merged['prev_latitude']) & 
    (df_merged['longitude'] == df_merged['prev_longitude']), 'status'] = 'poi'

# Drop the temporary columns used for calculation
df_merged = df_merged.drop(columns=['prev_latitude', 'prev_longitude'])
df_merged = df_merged.drop(df_merged.index[0])

In [ ]:
# 按小时分组，合并同一小时内重复坐标的记录
def merge_duplicates(df):
    # 按小时分组
    merged = []
    for hour, group in df.groupby('hours'):
        # 保留原始顺序
        group = group.reset_index(drop=True)
        i = 0
        while i < len(group):
            current_row = group.iloc[i].copy()
            current_lat, current_lon = current_row['latitude'], current_row['longitude']
            
            # 计算连续重复的记录数
            count = 1
            j = i + 1
            while j < len(group) and group.iloc[j]['latitude'] == current_lat and group.iloc[j]['longitude'] == current_lon:
                count += 1
                j += 1
            
            # 调整duration（乘以重复条数）
            current_row['duration'] = current_row['duration'] * count
            merged.append(current_row)
            
            # 跳过已处理的连续记录
            i = j
            
    return pd.DataFrame(merged)

id = "REDACTED_DEVICE_ID" # 一个移动很多的例子
# id = "REDACTED_DEVICE_ID"

df = data[data['device_aid'] == id]
df['duration'] = 60 / df.groupby('hours')['hours'].transform('count')
df['duration'] = df['duration'].round(2)
print(df.shape)
df_merged = merge_duplicates(df)
print(df_merged.shape)
df_merged['status'] = 'moving'  # Default all to moving

# Get previous row's coordinates (shift)
df_merged['prev_latitude'] = df_merged['latitude'].shift()
df_merged['prev_longitude'] = df_merged['longitude'].shift()
df_merged.loc[(df_merged['duration'] > 5), 'status'] = 'poi'
# If coordinates are the same as previous hour, mark as 'poi'
df_merged.loc[(df_merged['latitude'] == df_merged['prev_latitude']) & 
    (df_merged['longitude'] == df_merged['prev_longitude']), 'status'] = 'poi'

# Drop the temporary columns used for calculation
df_merged = df_merged.drop(columns=['prev_latitude', 'prev_longitude'])
df_merged = df_merged.drop(df_merged.index[0])

In [ ]:
location_poi_label = pd.DataFrame(df_merged[df_merged['status'] == 'poi'].value_counts(['latitude','longitude'])).reset_index()
location_poi_label['label'] = 'amenity'
location_poi_label.loc[0, 'label'] = 'home'
location_poi_label.loc[1, 'label'] = 'workplace'
df_merged = df_merged.merge(location_poi_label, on=['latitude', 'longitude'], how='left')